In [ ]:
import json
import os

import agama
import gizmo_analysis as gizmo
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


In [ ]:
sim_dir = "/Users/z5114326/Documents/simulations/"
# sim_dir = "/Volumes/Expansion/simulations/"
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

# from https://iopscience.iop.org/article/10.3847/2041-8213/ae6f14
# Dawn of the Milky Way Disk: Determination of When a Rotationally Supported Disk Appears and
# Dating the Spin-up of the Disk
MW_spin_up_lbt = 12.1  # lbt
MW_spin_up_err = 2.8  # lbt

# from BK23
MW_frac_avg = 0.30  # check this
MW_frac_low = 0.30 - 0.15  # check this
MW_frac_upp = 0.70 - 0.30  # check this
yerr = [[MW_frac_low], [MW_frac_upp]]

public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
timez0 = float(pub_data[pub_data["index"] == 600]["time_Gyr"].values[0])

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

su_path = "../data/sim_data/spin_up_lbt.json"
with open(su_path, "r", encoding="utf-8") as f:
    su_dict = json.load(f)


This is total Aurora population, need to confine to just fraction born in clusters

From other paper we have spin up times, go find stellar mass at z=0 born prior to that and born in the MW and then find fraction born in GCs. Do as a fraction of formation mass. Also dont forget to replace mass with GC mass. 

In [ ]:
def to_json_serializable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    else:
        return obj


def power_law(lbt, A, alpha):
    return A * (timez0 - lbt) ** (-alpha)

In [ ]:
rf_lim = 30  # kpc
r0_lim = 20  # kpc
max_masss = 10 ** np.arange(4, 7.1, 0.1)

save_max_json = "../data/sim_data/max_proto_fracs.json"
if os.path.exists(save_max_json):
    print("Loading proto_fracs.json")
    with open(save_max_json, "r", encoding="utf-8") as f:
        max_mass_dict = json.load(f)

else:
    max_mass_dict = {sim: {"mass": [], "frac": []} for sim in sim_lst}

    for sim in sim_lst:
        fire_dir = sim_dir + sim + "/" + sim + "_res7100"
        part = gizmo.io.Read.read_snapshots(
            # ["star", "dark"],
            ["star"],
            "index",
            600,
            fire_dir,
            assign_hosts_rotation=True,
            assign_formation_coordinates=True,
        )

        pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (600, 600)
        pot_nbody = agama.Potential(pot_file)

        ########################################################################

        rfmsk_s = part["star"].prop("form.host.distance.total") < rf_lim * part["star"].prop(
            "form.scalefactor"
        )
        r0msk_s = part["star"].prop("host.distance.total") < r0_lim
        tfmsk_s = part["star"].prop("form.time") < timez0 - su_dict[sim]

        msk_s = rfmsk_s & r0msk_s & tfmsk_s

        mass_prot = np.sum(part["star"]["mass"][msk_s])
        print(sim, mass_prot * 1e-10)

        ########################################################################

        ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
        ghost_data = h5py.File(ghost_file, "r")

        # it_id = "it001"
        for it_id in ghost_data.keys():
            src_dat = ghost_data[it_id]["source"]

            ptypes = np.array([ptype.decode("utf-8") for ptype in src_dat["ptype"][()]])
            pidxs = src_dat["pidx_snap600"][()]

            mfors = 10 ** src_dat["logm_tfor"][()]
            mfor_evs = 0.55 * mfors
            mz0 = np.array([0 if m0 == -1 else 10**m0 for m0 in src_dat["logm_tz0"][()]])
            mdifs = mfor_evs - mz0

            rz0s = np.linalg.norm(src_dat["pxyz_snap600"][()], axis=1)
            tfors = src_dat["tfor"][()]
            grps = src_dat["grpid"][()]

            ########################################################################
            # Apply selection criteria to GCs

            pt_star_mass = []
            pt_dark_mass = []

            gc_star_mass = []
            gc_dark_mass = []

            # --- loop ---
            for ptype, pidx, mdif, rz0, tfor, grp in zip(ptypes, pidxs, mdifs, rz0s, tfors, grps):
                if (rz0 < r0_lim) and (tfor < timez0 - su_dict[sim]) and (grp == 0):
                    if ptype == "star":
                        pt_star_mass.append(part["star"]["mass"][pidx])
                        gc_star_mass.append(mdif)
                    else:
                        # pt_dark_mass.append(part["dark"]["mass"][pidx])
                        gc_dark_mass.append(mdif)

            mass_prot_corr = mass_prot - np.sum(pt_star_mass) + np.sum(gc_star_mass) + np.sum(gc_dark_mass)

            # --- masks ---
            mass_arr = []
            frac_arr = []
            for max_mass in max_masss:
                mmsk = mfors < max_mass
                rmsk = rz0s < r0_lim
                tmsk = tfors < timez0 - su_dict[sim]
                gmsk = grps == 0

                msk = mmsk & rmsk & tmsk & gmsk
                mass = np.sum(mdifs[msk])

                mass_arr.append(mass)
                frac_arr.append(mass / mass_prot_corr)
            mass_arr = np.array(mass_arr)
            frac_arr = np.array(frac_arr)

            max_mass_dict[sim]["mass"].append(mass_arr)
            max_mass_dict[sim]["frac"].append(frac_arr)

    max_mass_dict_json = to_json_serializable(max_mass_dict)
    with open(save_max_json, "w", encoding="utf-8") as f:
        json.dump(max_mass_dict_json, f, indent=4)


In [ ]:
min_masss = 10 ** np.arange(4, 7.1, 0.1)

save_min_json = "../data/sim_data/min_proto_fracs.json"
if os.path.exists(save_min_json):
    print("Loading proto_fracs.json")
    with open(save_min_json, "r", encoding="utf-8") as f:
        min_mass_dict = json.load(f)

else:
    min_mass_dict = {sim: {"mass": [], "frac": []} for sim in sim_lst}

    for sim in sim_lst:
        fire_dir = sim_dir + sim + "/" + sim + "_res7100"
        part = gizmo.io.Read.read_snapshots(
            # ["star", "dark"],
            ["star"],
            "index",
            600,
            fire_dir,
            assign_hosts_rotation=True,
            assign_formation_coordinates=True,
        )

        pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (600, 600)
        pot_nbody = agama.Potential(pot_file)

        ########################################################################

        rfmsk_s = part["star"].prop("form.host.distance.total") < rf_lim * part["star"].prop(
            "form.scalefactor"
        )
        r0msk_s = part["star"].prop("host.distance.total") < r0_lim
        tfmsk_s = part["star"].prop("form.time") < timez0 - su_dict[sim]

        msk_s = rfmsk_s & r0msk_s & tfmsk_s

        mass_prot = np.sum(part["star"]["mass"][msk_s])
        print(sim, mass_prot * 1e-10)

        ########################################################################

        ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
        ghost_data = h5py.File(ghost_file, "r")

        # it_id = "it001"
        for it_id in ghost_data.keys():
            src_dat = ghost_data[it_id]["source"]

            ptypes = np.array([ptype.decode("utf-8") for ptype in src_dat["ptype"][()]])
            pidxs = src_dat["pidx_snap600"][()]

            mfors = 10 ** src_dat["logm_tfor"][()]
            mfor_evs = 0.55 * mfors
            mz0 = np.array([0 if m0 == -1 else 10**m0 for m0 in src_dat["logm_tz0"][()]])
            mdifs = mfor_evs - mz0

            rz0s = np.linalg.norm(src_dat["pxyz_snap600"][()], axis=1)
            tfors = src_dat["tfor"][()]
            grps = src_dat["grpid"][()]

            ########################################################################
            # Apply selection criteria to GCs

            pt_star_mass = []
            pt_dark_mass = []

            gc_star_mass = []
            gc_dark_mass = []

            # --- loop ---
            for ptype, pidx, mdif, rz0, tfor, grp in zip(ptypes, pidxs, mdifs, rz0s, tfors, grps):
                if (rz0 < r0_lim) and (tfor < timez0 - su_dict[sim]) and (grp == 0):
                    if ptype == "star":
                        pt_star_mass.append(part["star"]["mass"][pidx])
                        gc_star_mass.append(mdif)
                    else:
                        # pt_dark_mass.append(part["dark"]["mass"][pidx])
                        gc_dark_mass.append(mdif)

            mass_prot_corr = mass_prot - np.sum(pt_star_mass) + np.sum(gc_star_mass) + np.sum(gc_dark_mass)

            # --- masks ---
            mass_arr = []
            frac_arr = []
            for min_mass in min_masss:
                mmsk = mfors > min_mass
                rmsk = rz0s < r0_lim
                tmsk = tfors < timez0 - su_dict[sim]
                gmsk = grps == 0

                msk = mmsk & rmsk & tmsk & gmsk
                mass = np.sum(mdifs[msk])

                mass_arr.append(mass)
                frac_arr.append(mass / mass_prot_corr)
            mass_arr = np.array(mass_arr)
            frac_arr = np.array(frac_arr)

            min_mass_dict[sim]["mass"].append(mass_arr)
            min_mass_dict[sim]["frac"].append(frac_arr)

    min_mass_dict_json = to_json_serializable(min_mass_dict)
    with open(save_min_json, "w", encoding="utf-8") as f:
        json.dump(min_mass_dict_json, f, indent=4)


In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 6))

for sim in sim_lst:
    frac_avg = np.mean(max_mass_dict[sim]["frac"], axis=0)[-1]
    frac_std = np.std(max_mass_dict[sim]["frac"], axis=0)[-1]

    ax.scatter(su_dict[sim], frac_avg, color=sim_colors[sim], label=sim)
    ax.errorbar(su_dict[sim], frac_avg, yerr=frac_std, fmt="", capsize=5, color=sim_colors[sim])


ax.scatter(MW_spin_up_lbt, MW_frac_avg, color="magenta", label="Milky Way", marker="*", s=50)
ax.errorbar(
    MW_spin_up_lbt,
    MW_frac_avg,
    xerr=MW_spin_up_err,
    yerr=yerr,
    fmt="",
    capsize=5,
    color="magenta",
)

############################################################################################################

x_dat = [su_dict[sim] for sim in sim_lst]
x_dat.append(MW_spin_up_lbt)

y_dat = [np.mean(max_mass_dict[sim]["frac"], axis=0)[-1] for sim in sim_lst]
y_dat.append(MW_frac_avg)

y_err = [np.std(max_mass_dict[sim]["frac"], axis=0)[-1] for sim in sim_lst]
y_err.append(np.mean([MW_frac_low, MW_frac_upp]))

# Power-law fit
popt_pow, pcov_pow = curve_fit(
    power_law,
    x_dat,
    y_dat,
    p0=[1.0, 1.0],
    sigma=y_err,
    absolute_sigma=True,
)

A, alpha = popt_pow

t_fit = np.linspace(5, timez0 - 0.5, 200)
y_fit = power_law(t_fit, *popt_pow)

t = timez0 - t_fit

dy_dA = t ** (-alpha)
dy_dalpha = -A * t ** (-alpha) * np.log(t)

# Variance of y at each t
y_var = dy_dA**2 * pcov_pow[0, 0] + dy_dalpha**2 * pcov_pow[1, 1] + 2 * dy_dA * dy_dalpha * pcov_pow[0, 1]
y_std = np.sqrt(y_var)

# Plot line + shaded region
# ax.plot(t_fit, y_fit, label="Power-law fit", ls=":", c="r", zorder=0)
ax.fill_between(t_fit, y_fit - y_std, y_fit + y_std, color="r", alpha=0.1, label="Power-Law Fit")

# # ############################################################################################################


ax.set_xlim(timez0 - 0.5, 5)
ax.set_ylim(0, 0.75)

ax.set_xlabel("Spin-Up Lookback Time (Gyr)")
ax.set_ylabel("Fraction of Protogalaxy Stellar Mass with GC Origins")

# ax.set_yscale("log")

ax.legend()

save_loc = "../data/figures/galaxy_plots/proto_galaxy_fracion.pdf"
plt.savefig(save_loc)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 6))

for sim in sim_lst:
    frac_avg = 100 * np.mean(max_mass_dict[sim]["frac"], axis=0)[-1]
    frac_std = 100 * np.std(max_mass_dict[sim]["frac"], axis=0)[-1]

    ax.scatter(su_dict[sim], frac_avg, color=sim_colors[sim], label=sim)
    ax.errorbar(
        su_dict[sim],
        frac_avg,
        yerr=frac_std,
        fmt="",
        capsize=5,
        color=sim_colors[sim],
    )

# Milky Way
MW_frac_avg_pct = 100 * MW_frac_avg
MW_frac_low_pct = 100 * MW_frac_low
MW_frac_upp_pct = 100 * MW_frac_upp
yerr = [[MW_frac_low_pct], [MW_frac_upp_pct]]

ax.scatter(
    MW_spin_up_lbt,
    MW_frac_avg_pct,
    color="magenta",
    label="Milky Way",
    marker="*",
    s=50,
)
ax.errorbar(
    MW_spin_up_lbt,
    MW_frac_avg_pct,
    xerr=MW_spin_up_err,
    yerr=yerr,
    fmt="",
    capsize=5,
    color="magenta",
)

############################################################################################################

x_dat = [su_dict[sim] for sim in sim_lst]
x_dat.append(MW_spin_up_lbt)

y_dat = [100 * np.mean(max_mass_dict[sim]["frac"], axis=0)[-1] for sim in sim_lst]
y_dat.append(MW_frac_avg_pct)

y_err = [100 * np.std(max_mass_dict[sim]["frac"], axis=0)[-1] for sim in sim_lst]
y_err.append(np.mean([MW_frac_low_pct, MW_frac_upp_pct]))

# Fit (fit in fractional units)
popt_pow, pcov_pow = curve_fit(
    power_law,
    x_dat,
    np.array(y_dat) / 100,
    p0=[1.0, 1.0],
    sigma=np.array(y_err) / 100,
    absolute_sigma=True,
)

A, alpha = popt_pow

t_fit = np.linspace(5, timez0 - 0.5, 200)
y_fit = 100 * power_law(t_fit, *popt_pow)

t = timez0 - t_fit

dy_dA = t ** (-alpha)
dy_dalpha = -A * t ** (-alpha) * np.log(t)

y_var = dy_dA**2 * pcov_pow[0, 0] + dy_dalpha**2 * pcov_pow[1, 1] + 2 * dy_dA * dy_dalpha * pcov_pow[0, 1]
y_std = 100 * np.sqrt(y_var)

ax.fill_between(
    t_fit,
    y_fit - y_std,
    y_fit + y_std,
    color="r",
    alpha=0.1,
    label="Power-Law Fit",
)

############################################################################################################

ax.set_xlim(timez0 - 0.5, 5)
ax.set_ylim(0, 75)

ax.set_xlabel("Spin-Up Lookback Time (Gyr)")
ax.set_ylabel("Percentage of Protogalaxy Stellar Mass with GC Origins (%)")

ax.legend()

save_loc = "../data/figures/galaxy_plots/proto_galaxy_percentage.pdf"
plt.savefig(save_loc)

In [ ]:
proto_dict = {sim: {"t_mass": [], "p_mass": []} for sim in sim_lst}

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    # it_id = "it001"
    for it_id in ghost_data.keys():
        src_dat = ghost_data[it_id]["source"]

        ptypes = np.array([ptype.decode("utf-8") for ptype in src_dat["ptype"][()]])
        pidxs = src_dat["pidx_snap600"][()]

        mfors = 10 ** src_dat["logm_tfor"][()]
        mfor_evs = 0.55 * mfors
        mz0 = np.array([0 if m0 == -1 else 10**m0 for m0 in src_dat["logm_tz0"][()]])
        mdifs = mfor_evs - mz0

        rz0s = np.linalg.norm(src_dat["pxyz_snap600"][()], axis=1)
        tfors = src_dat["tfor"][()]
        grps = src_dat["grpid"][()]

        # --- masks ---

        rmsk = rz0s < r0_lim
        tmsk = tfors < timez0 - su_dict[sim]
        gmsk = grps == 0

        t_msk = rmsk & gmsk
        t_mass = np.sum(mdifs[t_msk])

        p_msk = rmsk & tmsk & gmsk
        p_mass = np.sum(mdifs[p_msk])

        proto_dict[sim]["t_mass"].append(t_mass)
        proto_dict[sim]["p_mass"].append(p_mass)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for sim in sim_lst:
    t_mass_mean = np.mean(proto_dict[sim]["t_mass"])
    t_mass_std = np.std(proto_dict[sim]["t_mass"])

    p_mass_mean = np.mean(proto_dict[sim]["p_mass"])
    p_mass_std = np.std(proto_dict[sim]["p_mass"])

    ax.barh(sim, t_mass_mean, color=sim_colors[sim], alpha=0.2)
    ax.barh(sim, p_mass_mean, color=sim_colors[sim], alpha=1.0)


ax.invert_yaxis()
ax.set_xscale("log")

ax.set_xlim(7 * 10**7, 4 * 10**8)
ax.set_xlabel(r"Disrupted Cluster Mass at z = 0 [M$_{\odot}$]")

save_loc = "../data/figures/galaxy_plots/proto_galaxy_gc_mass.pdf"
plt.savefig(save_loc)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(6, 8))
plt.subplots_adjust(hspace=0.05)

for sim in sim_lst:
    mass_avg = np.mean(min_mass_dict[sim]["mass"], axis=0)
    mass_std = np.std(min_mass_dict[sim]["mass"], axis=0)

    frac_avg = np.mean(min_mass_dict[sim]["frac"], axis=0)
    frac_std = np.std(min_mass_dict[sim]["frac"], axis=0)

    #####################################################################

    axs[0].plot(min_masss, mass_avg, c=sim_colors[sim], label=sim)
    axs[0].fill_between(min_masss, mass_avg + mass_std, mass_avg - mass_std, color=sim_colors[sim], alpha=0.1)
    axs[0].set_xlim(10**4, 7 * 10**6)
    axs[0].set_xscale("log")
    axs[0].set_xticklabels([])

    axs[0].set_ylim(5 * 10**4, 4 * 10**9)
    axs[0].set_yscale("log")
    axs[0].set_ylabel(r"Protogalaxy Mass Originating in GCs [M$_{\odot}$]")

    axs[0].legend(loc="lower left")

    #####################################################################

    axs[1].plot(min_masss, frac_avg, c=sim_colors[sim], label=sim)
    axs[1].fill_between(min_masss, frac_avg + frac_std, frac_avg - frac_std, color=sim_colors[sim], alpha=0.1)
    axs[1].set_xlim(10**4, 7 * 10**6)
    axs[1].set_xscale("log")
    axs[1].set_xlabel(r"Minimum GC Mass [M$_{\odot}$]")
    # axs[1].set_xlabel(r"Upper Limit of Initial GC Mass Included [M$_{\odot}$]")

    axs[1].set_ylim(4 * 10**-6, 2 * 10**0)
    axs[1].set_yscale("log")
    axs[1].set_ylabel(r"Fraction of Protogalaxy Stellar Mass with GC Origins")

#####################################################################
#####################################################################
# MW Additions
mw_text = "Milky Way Aurora Population"

# From BK23
f_disrupt = 2 / 3  # example value from the text
factor = 1 / (1 - f_disrupt)
min_mass = (5 - 2) * 1e8 * factor
max_mass = (5 + 2) * 1e8 * factor

f_low, f_high = 0.15, 0.70
GC_mass_min = f_low * min_mass
GC_mass_max = f_high * max_mass

axs[0].fill_between(max_masss, GC_mass_min, GC_mass_max, color="r", alpha=0.1, zorder=0)
axs[0].axhline(GC_mass_min, c="r", ls="--", lw=0.5)
axs[0].axhline(GC_mass_max, c="r", ls="--", lw=0.5)
axs[0].text(0.5, 0.80, mw_text, transform=axs[0].transAxes, color="r", ha="center", va="center")

axs[1].fill_between(max_masss, f_low, f_high, color="r", alpha=0.1, zorder=0)
axs[1].axhline(f_low, c="r", ls="--", lw=0.5)
axs[1].axhline(f_high, c="r", ls="--", lw=0.5)
axs[1].text(0.5, 0.85, mw_text, transform=axs[1].transAxes, color="r", ha="center", va="center")

save_loc = "../data/figures/galaxy_plots/proto_min_gc_mass.pdf"
plt.savefig(save_loc)